# ⚛️ Hackathon Algorithmique Quantique : Optimisation de l'Énergie Moléculaire

Bienvenue dans ce challenge ! L'objectif de ce hackathon est de trouver l'énergie de l'état fondamental de différentes molécules à l'aide d'algorithmes quantiques variationnels (comme le VQE). 

Dans ce notebook, nous allons :
1. Définir nos molécules (Chaîne d'hydrogène et Hydrure de Lithium - LiH).
2. Générer les représentations fermioniques de ces molécules, avec et sans "Frozen Core" (gel du cœur).
3. Établir une ligne de base (Baseline) avec un VQE classique (Ansatz UCCSD).
4. **À vous de jouer :** Compléter la fonction `run()` pour battre notre temps d'exécution tout en conservant (ou en améliorant) la précision chimique de l'énergie !

## [Copie de test — équipe] Ce qui a été ajouté à la consigne

Copie de `main.ipynb` remplie avec notre méthode (k-ADAPT-VQE auto-ajusté, meilleure configuration du run 3). Les ajouts sont marqués **[AJOUT ÉQUIPE]** / *[Ajout équipe]*.

0. **Installation** : cellule `pip install` (qiskit, qiskit-nature, qiskit-algorithms, pyscf) avant les imports, pour Colab.
1. **§1 Molécule** : molécule de test sélectionnable `MOLECULE` (défaut **HF**, fluorure d'hydrogène : 10 qubits gelé / 12 complet, 8 électrons) ; `LiH` (consigne) et `H4` disponibles.
2. **§2** : les drivers utilisent `mol_info` au lieu de `lih_info` ; affichage du nombre de qubits.
3. **§3** : correction d'un bug de la consigne (`exact_energy` utilisé au §4 et §6 mais jamais défini → `exact_energy = exact_energy_frozen`) et des libellés.
4. **§5** : description de la méthode, cellule « moteur » k-ADAPT (pool, simulateur, k-ADAPT, circuit Qiskit, comptage des portes), et fonction `run()` remplie (signature et retour inchangés).
5. **§6** : cellule d'évaluation de la consigne **inchangée**, puis **bilan selon nos priorités** (précision chimique → gain en temps vs VQE classique → CNOT / portes / profondeur, comptés avec la même transpilation pour les deux méthodes) et courbe de convergence.
6. **§7** : `run()` sur le problème complet (non gelé), comparé à `exact_energy_full`.

In [ ]:
# [AJOUT ÉQUIPE] Installation des dépendances (Colab ne fournit ni Qiskit Nature, ni Qiskit Algorithms, ni PySCF).
# Versions testées ensemble. À exécuter une fois, AVANT la cellule d'imports.
import sys, subprocess, importlib.util
_PKGS = {"qiskit": "qiskit==2.6.2", "qiskit_nature": "qiskit-nature==0.8.0",
         "qiskit_algorithms": "qiskit-algorithms==0.4.0", "pyscf": "pyscf==2.15.0"}
_missing = [spec for mod, spec in _PKGS.items() if importlib.util.find_spec(mod) is None]
if _missing:
    print("Installation :", ", ".join(_missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    importlib.invalidate_caches()
print("Dépendances OK")

In [ ]:
import time
import numpy as np

# Qiskit et Qiskit Nature
from qiskit_nature.second_q.formats.molecule_info import MoleculeInfo
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.transformers import FreezeCoreTransformer
from qiskit_nature.second_q.mappers import JordanWignerMapper, ParityMapper
from qiskit_nature.second_q.algorithms import GroundStateEigensolver
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD

from qiskit_algorithms import VQE, NumPyMinimumEigensolver
from qiskit_algorithms.optimizers import SLSQP
from qiskit.primitives import StatevectorEstimator as Estimator

## 1. Définition de la Molécule

Nous utilisons la classe `MoleculeInfo` pour définir la géométrie de nos molécules. 
Nous vous fournissons une fonction pour générer une chaîne d'atomes d'Hydrogène, ainsi que la configuration pour la molécule d'Hydrure de Lithium (LiH) qui servira pour notre calcul de référence.

In [ ]:
def build_hydrogen_chain(num_atoms, bond_distance):
    symbols = ["H"] * num_atoms
    coords = [(i * bond_distance, 0.0, 0.0) for i in range(num_atoms)]
    multiplicity = (num_atoms % 2) + 1 
    return MoleculeInfo(symbols=symbols, coords=coords, multiplicity=multiplicity, charge=0)

# Définition de H2 (chaine de 2 atomes)
h2_info = build_hydrogen_chain(num_atoms=2, bond_distance=0.735)

# Définition de la molécule LiH pour notre Baseline
lih_info = MoleculeInfo(
    symbols=["Li", "H"],
    coords=[(0.0, 0.0, 0.0), (0.0, 0.0, 1.5474)],
    multiplicity=1,
    charge=0
)

# ==========================================================================================
# [AJOUT ÉQUIPE] Molécule de test, sélectionnable. Défaut : fluorure d'hydrogène (HF).
#  - HF gelé : 5 orbitales, 8 électrons -> 10 qubits (comme LiH) ; complet : 12 qubits (< NH3 : 16).
#    Même taille de circuit que LiH, mais 8 électrons au lieu de 2 : chimiquement plus riche.
#  - H2O / BeH2 (12 qubits gelés) : à éviter AVEC LA BASELINE DE LA CONSIGNE -> UCCSD à 92 paramètres,
#    ~150 s par évaluation d'énergie mesuré (SLSQP en fait ~93 par itération) : la baseline ne finit pas.
# ==========================================================================================
hf_info = MoleculeInfo(
    symbols=["H", "F"],
    coords=[(0.0, 0.0, 0.0), (0.0, 0.0, 0.917)],   # longueur de liaison d'équilibre (Å)
    multiplicity=1,
    charge=0
)
h4_info = build_hydrogen_chain(num_atoms=4, bond_distance=0.735)   # 8 qubits, rien à geler

MOLECULE = "HF"   # "HF", "LiH" (consigne), "H4"
mol_info = {"HF": hf_info, "LiH": lih_info, "H4": h4_info}[MOLECULE]
print(f"Molécule testée : {MOLECULE}")

## 2. Génération de la molécule (PySCF) & Frozen Core

Nous utilisons le driver `PySCF` pour calculer les intégrales à 1 et 2 corps. 
Afin de vous permettre de vous entraîner sur des tailles d'espace de Hilbert différentes mais chimiquement identiques, nous générons deux versions du problème LiH :
- **Complète** : Tous les électrons sont pris en compte.
- **Gelée (Frozen Core)** : Les électrons de cœur (qui participent très peu aux liaisons chimiques) sont ignorés, réduisant ainsi drastiquement le nombre de qubits nécessaires.

In [ ]:
# --- Molécule sans geler le coeur ---
driver_full = PySCFDriver.from_molecule(mol_info, basis="sto3g")
problem_full = driver_full.run()
fermionic_op_full = problem_full.hamiltonian.second_q_op()

# --- Molécule en gelant le coeur (Frozen Core) ---
driver_frozen = PySCFDriver.from_molecule(mol_info, basis="sto3g")
problem_frozen = driver_frozen.run()

# Application du FreezeCoreTransformer
transformer = FreezeCoreTransformer(freeze_core=True)
problem_frozen = transformer.transform(problem_frozen)
fermionic_op_frozen = problem_frozen.hamiltonian.second_q_op()

print(f"Nombre d'orbitales spatiales (Full) : {problem_full.num_spatial_orbitals}")
print(f"Nombre d'orbitales spatiales (Frozen) : {problem_frozen.num_spatial_orbitals}")
print(f"Qubits (Jordan-Wigner) : {2 * problem_full.num_spatial_orbitals} (complet), {2 * problem_frozen.num_spatial_orbitals} (gelé) | électrons actifs (gelé) : {problem_frozen.num_particles}")

## 3. Calcul des valeurs propres (Énergie Exacte)

Avant de lancer nos algorithmes quantiques, nous devons connaître la "vraie" réponse. Nous utilisons un solveur classique (`NumPyMinimumEigensolver`) pour diagonaliser exactement le Hamiltonien et trouver l'énergie de l'état fondamental de notre molécule LiH gelée. 
Cette valeur servira à calculer la précision de vos méthodes.

In [ ]:
mapper = JordanWignerMapper()
exact_solver = NumPyMinimumEigensolver()
exact_calc = GroundStateEigensolver(mapper, exact_solver)

exact_result_frozen = exact_calc.solve(problem_frozen)
exact_energy_frozen = exact_result_frozen.total_energies[0]

exact_result_full = exact_calc.solve(problem_full)
exact_energy_full = exact_result_full.total_energies[0]

print(f"Énergie totale exacte de l'état fondamental ({MOLECULE} gelé) : {exact_energy_frozen:.6f} Hartree")

print(f"Énergie totale exacte de l'état fondamental ({MOLECULE} complet) : {exact_energy_full:.6f} Hartree")

# [AJOUT ÉQUIPE] la consigne utilise `exact_energy` (non défini -> NameError) : la référence est le problème gelé
exact_energy = exact_energy_frozen

## 4. Baseline VQE UCCSD

Nous lançons maintenant un VQE classique sur la molécule LiH gelée.
- **Ansatz** : UCCSD (Unitary Coupled-Cluster Singles and Doubles).
- **État initial** : Tous les paramètres fixés à 0 (partant de l'état Hartree-Fock).
- **Objectif** : Mesurer le temps d'exécution (`t_ref`) et vérifier l'énergie. **C'est ce temps que vous devrez battre !**

> ⏱️ *[Ajout équipe]* Cette baseline est lente : ~5 s par évaluation d'énergie sur LiH/HF gelés (24 paramètres) et SLSQP fait ~25 évaluations par itération (différences finies) → compter plusieurs dizaines de minutes.

In [ ]:
# Configuration de l'état de référence Hartree-Fock
initial_state = HartreeFock(
    problem_frozen.num_spatial_orbitals,
    problem_frozen.num_particles,
    mapper
)

# Configuration de l'Ansatz UCCSD
ansatz = UCCSD(
    problem_frozen.num_spatial_orbitals,
    problem_frozen.num_particles,
    mapper,
    initial_state=initial_state
)

# Initialisation de tous les paramètres à 0
initial_pt = np.zeros(ansatz.num_parameters)

# Setup du VQE
estimator = Estimator()
optimizer = SLSQP(maxiter=100)
vqe = VQE(estimator, ansatz, optimizer, initial_point=initial_pt)
vqe_solver = GroundStateEigensolver(mapper, vqe)

print("Lancement du VQE UCCSD de référence. Veuillez patienter...")
start_time = time.time()
baseline_result = vqe_solver.solve(problem_frozen)
end_time = time.time()

t_ref = end_time - start_time
baseline_energy = baseline_result.total_energies[0]
precision_baseline = abs(exact_energy - baseline_energy)

print("-" * 50)
print(f"⏱️ Temps de référence (t_ref) : {t_ref:.2f} secondes")
print(f"⚛️ Énergie calculée (Baseline) : {baseline_energy:.6f} Hartree")
print(f"🎯 Précision (Erreur) absolue : {precision_baseline:.6f} Hartree")
print("-" * 50)

## 5. Le Hackathon : Votre fonction `run()`

C'est ici que vous intervenez. Vous devez implémenter la fonction `run()`. 

**Instructions :**
1. Elle prend en entrée l'objet `problem` (pour récupérer le nombre de particules/orbitales si besoin) et l'opérateur `fermionic_op` (sortie du PySCF driver).
2. **Mapper** : Appliquez le mapper de votre choix (`ParityMapper`, `JordanWignerMapper`, `BravyiKitaevMapper`...) pour transformer la représentation fermionique sous la forme d'une somme de matrices de Pauli (qubit operator).
3. **Méthode** : Configurez votre propre ansatz, état initial (ex: paramétrage aléatoire ou hardware-efficient) et optimiseur.
4. **Retour** : La fonction doit retourner l'énergie totale (nucléaire incluse) trouvée et le temps d'exécution.

> 💡 *Astuce : Pour gagner en temps, explorez des ansatz plus légers que UCCSD (comme `EfficientSU2`), des mappers qui réduisent le nombre de qubits (comme `ParityMapper` avec réduction de symétrie), ou des optimiseurs plus agressifs.*

### [Ajout équipe] Notre méthode : k-ADAPT-VQE auto-ajusté

Configuration = **meilleure configuration trouvée par l'auto-tuner** (run 3, `run_3/best_config.json`, `autotune-v6`), choisie selon nos priorités : **1) précision chimique** (contrainte dure : erreur < 0,25 × 1,6 mHa à l'entraînement), **2) gain vs VQE classique**, **3) nombre de portes**.

| Élément | Choix |
|---|---|
| Mapper | Jordan-Wigner (la parité réduit 2 qubits mais multiplie les CNOT par ~5-7 : écarté au run 3) |
| Ansatz | construit **dynamiquement** à partir de Hartree-Fock (paramètres initiaux à 0) avec le pool **QEB** (excitations *qubit* simples / doubles : 2 / 14 CNOT) |
| Sélection | k dynamique : à chaque itération, tous les opérateurs de score ≥ `alpha`=0,5 × meilleur score, au plus `k_max`=5 ; score = \|gradient\| / CNOT^`beta` avec `beta`=0,5 (préfère l'opérateur bon marché) |
| Optimiseur | BFGS + gradients exacts (méthode adjointe), départ à chaud |
| Arrêt | norme du gradient du pool < `eps` = 5·10⁻⁴ |
| Élagage | protégé : retire les \|θ\| < 0,02 si l'énergie remonte de moins de 0,05 mHa |
| Simulation | vecteur d'état restreint au secteur (N_α, N_β) — les excitations le conservent, c'est le même circuit simulé plus efficacement |
| Énergie finale | **mesurée sur le vrai circuit Qiskit** (portes natives) avec `Estimator`, + constantes (répulsion nucléaire, énergie du cœur gelé) |

In [ ]:
# ==========================================================================================
# k-ADAPT-VQE auto-ajusté (notre méthode) -- version Qiskit autonome
# Configuration = meilleure config de l'auto-tuner (run_3/best_config.json, run_tag autotune-v6)
# ==========================================================================================
from itertools import combinations
from scipy.optimize import minimize
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import UnitaryGate, UCRYGate

KADAPT_CFG = dict(pool="QEB", k_max=5, alpha=0.5, beta=0.5, eps=5e-4, plateau=0.0, prune_tol=0.02)
GAIN_MIN = 1e-9          # gain d'énergie (Ha) en dessous duquel un ajout est jugé inutile
PRUNE_MAX_LOSS = 5e-5    # élagage protégé : perte d'énergie max acceptée (0,05 mHa)
MAX_OPS = 150            # taille max de l'ansatz
BFGS_GTOL = 1e-6
CNOT_COST = {"S": 2, "D": 14}   # CNOT par excitation qubit (simple / double) -> sélection sensible au coût (beta)
BASIS_GATES = ["cx", "rz", "sx", "x"]
RUN_INFO = {}            # détails du dernier appel à run() (ansatz, circuit, historique...)


# ---------- 1. Pool QEB (excitations qubit), ordre Qiskit Jordan-Wigner : alpha = 0..n-1, beta = n..2n-1 ----------
def qeb_pool(n, num_particles):
    na, nb = num_particles
    oa, va = list(range(na)), list(range(na, n))
    ob, vb = [n + i for i in range(nb)], [n + i for i in range(nb, n)]
    doubles = ([("D", i + a) for i in combinations(oa, 2) for a in combinations(va, 2)]
               + [("D", i + a) for i in combinations(ob, 2) for a in combinations(vb, 2)]
               + [("D", (i, j, a, b)) for i in oa for j in ob for a in va for b in vb])
    singles = [("S", (i, a)) for i in oa for a in va] + [("S", (i, a)) for i in ob for a in vb]
    return doubles + singles


# ---------- 2. Simulateur vecteur d'état restreint au secteur (N_alpha, N_beta) ----------
# Les excitations qubit conservent N_alpha et N_beta : on ne simule que ce sous-espace
# (ex. HF gelé : 25 états au lieu de 2^10). C'est exactement le même circuit, simulé plus efficacement.
def sector_basis(n, na, nb):
    sa = [sum(1 << q for q in c) for c in combinations(range(n), na)]
    sb = [sum(1 << (n + q) for q in c) for c in combinations(range(n), nb)]
    return np.array(sorted(a | b for a in sa for b in sb), dtype=np.int64)

def op_pairs(op, basis):
    # couples d'états (source, cible) reliés par l'excitation : rotation de Givens dans chaque couple
    kind, w = op
    src, dst = (w[:1], w[1:]) if kind == "S" else (w[:2], w[2:])
    ms, md = sum(1 << q for q in src), sum(1 << q for q in dst)
    p = np.where((basis & (ms | md)) == ms)[0]
    return p, np.searchsorted(basis, basis[p] ^ (ms | md))

def apply_op(psi, pq, t):
    p, q = pq
    a, b = psi[p], psi[q]
    psi[p], psi[q] = np.cos(t) * a - np.sin(t) * b, np.sin(t) * a + np.cos(t) * b

def energy_and_grad(H, psi0, pqs, x):
    # énergie + gradient exact par méthode adjointe (une passe avant, une passe arrière)
    psi = psi0.copy()
    for pq, t in zip(pqs, x):
        apply_op(psi, pq, t)
    lam = H @ psi
    E = float(psi @ lam)
    g = np.empty(len(x))
    for k in range(len(x) - 1, -1, -1):
        p, q = pqs[k]
        g[k] = 2.0 * (lam[q] @ psi[p] - lam[p] @ psi[q])
        apply_op(psi, pqs[k], -x[k]); apply_op(lam, pqs[k], -x[k])
    return E, g


# ---------- 3. k-ADAPT généralisé (k dynamique, sélection sensible au coût, élagage protégé) ----------
def kadapt(H, psi0, pool_pq, costs, E_hf, cfg=KADAPT_CFG, max_ops=MAX_OPS):
    stats = dict(n_E=0)
    def optimize(idx, x0):
        pqs = [pool_pq[i] for i in idx]
        def fun(x):
            stats["n_E"] += 1
            return energy_and_grad(H, psi0, pqs, x)
        r = minimize(fun, x0, jac=True, method="BFGS", options=dict(gtol=BFGS_GTOL, maxiter=500))
        return np.array(r.x), float(r.fun)

    sel, x, E = [], np.zeros(0), E_hf
    hist = [dict(n_ops=0, E=E)]
    stop = "max_ops"
    while len(sel) < max_ops:
        # gradients de tout le pool à theta = 0 : dE/dtheta_k = 2 <H psi | A_k psi>
        psi = psi0.copy()
        for i, t in zip(sel, x):
            apply_op(psi, pool_pq[i], t)
        lam = H @ psi
        g = np.array([2.0 * (lam[q] @ psi[p] - lam[p] @ psi[q]) for p, q in pool_pq])
        if np.linalg.norm(g) < cfg["eps"]:
            stop = "gradient"; break
        score = np.abs(g) / costs ** cfg["beta"]
        order = np.argsort(-score)
        chosen = [int(c) for c in order[:min(cfg["k_max"], max_ops - len(sel))]
                  if score[c] >= cfg["alpha"] * score[order[0]] and abs(g[c]) > 1e-10]
        new_sel = sel + chosen
        xn, En = optimize(new_sel, np.append(x, np.zeros(len(chosen))))
        gain = E - En
        if gain < max(GAIN_MIN, cfg["plateau"]):
            # opérateur "bon marché" choisi (beta>0) mais inutile : on retente avec le plus grand |gradient|
            c0 = int(np.argmax(np.abs(g)))
            if c0 not in chosen:
                x2, E2 = optimize(sel + [c0], np.append(x, 0.0))
                if E - E2 > gain:
                    new_sel, xn, En, gain = sel + [c0], x2, E2, E - E2
        if gain < max(GAIN_MIN, cfg["plateau"]):
            stop = "plateau"; break             # l'ajout ne fait rien gagner : on ne le garde pas
        sel, x, E = new_sel, xn, En
        hist.append(dict(n_ops=len(sel), E=E))

    # élagage PROTÉGÉ : retire les |theta| < prune_tol si l'énergie remonte de moins de PRUNE_MAX_LOSS
    if cfg["prune_tol"] > 0 and len(x):
        cand = [int(i) for i in np.argsort(np.abs(x)) if abs(x[i]) < cfg["prune_tol"]]
        while cand and len(cand) < len(x):
            keep = [i for i in range(len(sel)) if i not in cand]
            x2, E2 = optimize([sel[i] for i in keep], x[keep])
            if E2 - E <= PRUNE_MAX_LOSS:
                sel, x, E = [sel[i] for i in keep], x2, E2
                hist.append(dict(n_ops=len(sel), E=E, pruned=True))
                break
            cand = cand[:len(cand) // 2]
    return sel, x, E, hist, dict(arret=stop, n_eval_E=stats["n_E"])


# ---------- 4. Circuit Qiskit réel de l'ansatz (portes natives, utilisé pour l'énergie finale et le comptage) ----------
def single_excitation_gate(t):
    # rotation de Givens |1_i 0_a> <-> |0_i 1_a>  (matrice 4x4, synthèse optimale : 2 CNOT)
    U = np.eye(4); c, s = np.cos(t), np.sin(t)
    U[1, 1], U[1, 2], U[2, 1], U[2, 2] = c, -s, s, c
    return UnitaryGate(U, label="S")

def append_double_excitation(qc, t, i, j, a, b):
    # |1_i 1_j 0_a 0_b> <-> |0_i 0_j 1_a 1_b> : 3 CNOT + RY(2t) uniformément contrôlé (8 CNOT) + 3 CNOT = 14 CNOT
    qc.cx(a, b); qc.cx(i, j); qc.cx(a, i)
    angles = [0.0] * 8
    angles[1] = 2 * t                  # contrôles (i, j, b) = (1, 0, 0) ; i = bit de poids faible
    qc.append(UCRYGate(angles), [a, i, j, b])
    qc.cx(a, i); qc.cx(i, j); qc.cx(a, b)

def build_circuit(n, num_particles, ops, x):
    na, nb = num_particles
    qc = QuantumCircuit(2 * n)
    for q in list(range(na)) + [n + i for i in range(nb)]:
        qc.x(q)                          # état de Hartree-Fock
    for (kind, w), t in zip(ops, x):
        if kind == "S":
            qc.append(single_excitation_gate(t), list(w))
        else:
            append_double_excitation(qc, t, *w)
    return qc

def gate_counts(qc):
    tq = transpile(qc, basis_gates=BASIS_GATES, optimization_level=1)
    ops = tq.count_ops()
    return dict(n_cnot=int(ops.get("cx", 0)), n_portes=int(sum(ops.values())), profondeur=int(tq.depth()))

In [ ]:
def run(problem, fermionic_op):
    """
    Exécute votre algorithme quantique sur l'opérateur fermionique donné.
    Méthode : k-ADAPT-VQE auto-ajusté (pool QEB, k dynamique, sélection sensible au coût, élagage protégé).
    """
    # On démarre le chrono !
    start_time = time.time()

    # ==========================================
    # 🛠️ VOTRE CODE COMMENCE ICI 🛠️
    # ==========================================

    # 1. Mapper : Jordan-Wigner (le mapping de parité réduit 2 qubits mais multiplie les CNOT par ~5-7, cf. run_3)
    my_mapper = JordanWignerMapper()
    qubit_op = my_mapper.map(fermionic_op)

    # 2. Ansatz construit dynamiquement par k-ADAPT à partir de Hartree-Fock (paramètres initiaux à 0)
    n = problem.num_spatial_orbitals
    na, nb = problem.num_particles
    basis = sector_basis(n, na, nb)
    H = qubit_op.to_matrix(sparse=True)[basis][:, basis].real.tocsr()
    hf_index = sum(1 << q for q in list(range(na)) + [n + i for i in range(nb)])
    psi0 = np.zeros(len(basis))
    k_hf = int(np.searchsorted(basis, hf_index))
    psi0[k_hf] = 1.0
    pool = qeb_pool(n, (na, nb))
    pool_pq = [op_pairs(o, basis) for o in pool]
    costs = np.array([CNOT_COST[k] for k, _ in pool], dtype=float)

    # 3. VQE adaptatif : sélection d'opérateurs par gradient + optimiseur BFGS avec gradients exacts (adjoint)
    sel, x, e_internal, hist, info = kadapt(H, psi0, pool_pq, costs, float(H[k_hf, k_hf]))
    ops = [pool[i] for i in sel]

    # 4. Énergie finale MESURÉE sur le vrai circuit Qiskit (Estimator), + constantes (répulsion nucléaire, cœur gelé)
    qc = build_circuit(n, (na, nb), ops, x)
    e_elec = float(Estimator().run([(qc, qubit_op)]).result()[0].data.evs)
    constants = sum(problem.hamiltonian.constants.values())
    my_energy = e_elec + constants   # Énergie totale

    # ==========================================
    # 🏁 VOTRE CODE S'ARRÊTE ICI 🏁
    # ==========================================

    end_time = time.time()
    execution_time = end_time - start_time

    RUN_INFO.clear()
    RUN_INFO.update(n_qubits=qc.num_qubits, taille_pool=len(pool), ops=ops, params=x, circuit=qc,
                    hist=[dict(h, E=h["E"] + constants) for h in hist],
                    ecart_simulateur_circuit=abs(e_elec - e_internal), **info)
    return my_energy, execution_time

## 6. Évaluation

Testons votre algorithme sur la molécule LiH gelée et comparons vos résultats avec la Baseline !

In [ ]:
print("Lancement de votre fonction run()...")

# Exécution de votre fonction sur le problème gelé
participant_energy, participant_time = run(problem_frozen, fermionic_op_frozen)

# Calcul des métriques
participant_error = abs(exact_energy - participant_energy)
speedup = t_ref / participant_time if participant_time > 0 else float('inf')

print("\n" + "="*50)
print("🏆 RÉSULTATS DU HACKATHON 🏆")
print("="*50)
print(f"⏱️ Votre temps       : {participant_time:.2f} secondes (x{speedup:.2f} par rapport au t_ref)")
print(f"⚛️ Votre énergie     : {participant_energy:.6f} Hartree")
print(f"🎯 Votre précision   : {participant_error:.6f} Hartree d'erreur")
print("-" * 50)

# Conditions de victoire basiques (à adapter)
CHEMICAL_ACCURACY = 0.0016  # Précision chimique standard (~1 kcal/mol)

if participant_error <= CHEMICAL_ACCURACY:
    print("✅ Précision chimique ATTEINTE !")
else:
    print("❌ Précision chimique NON ATTEINTE. Essayez d'améliorer votre Ansatz ou votre optimiseur.")

In [ ]:
# [AJOUT ÉQUIPE] Bilan selon nos priorités : 1) précision chimique  2) gain vs VQE classique  3) nombre de portes
import pandas as pd
import matplotlib.pyplot as plt

res_base = gate_counts(ansatz)                 # UCCSD de la baseline (même transpilation, mêmes portes de base)
res_ours = gate_counts(RUN_INFO["circuit"])
bilan = pd.DataFrame([
    dict(methode="Baseline VQE UCCSD (SLSQP)", n_qubits=ansatz.num_qubits, n_params=ansatz.num_parameters,
         erreur_mHa=1e3 * precision_baseline, precision_chimique=precision_baseline <= CHEMICAL_ACCURACY,
         temps_s=t_ref, **res_base),
    dict(methode="Notre k-ADAPT auto-ajusté", n_qubits=RUN_INFO["n_qubits"], n_params=len(RUN_INFO["params"]),
         erreur_mHa=1e3 * participant_error, precision_chimique=participant_error <= CHEMICAL_ACCURACY,
         temps_s=participant_time, **res_ours),
]).set_index("methode")
display(bilan.round(4))

print(f"1) Précision chimique : {'✅' if participant_error <= CHEMICAL_ACCURACY else '❌'} "
      f"({1e3 * participant_error:.4f} mHa, seuil {1e3 * CHEMICAL_ACCURACY:.1f} mHa)")
print(f"2) Gain en temps vs VQE classique : x{speedup:.1f}")
print(f"3) Gain en CNOT : x{res_base['n_cnot'] / max(res_ours['n_cnot'], 1):.1f} "
      f"({res_base['n_cnot']} -> {res_ours['n_cnot']}), portes : x{res_base['n_portes'] / max(res_ours['n_portes'], 1):.1f}, "
      f"profondeur : x{res_base['profondeur'] / max(res_ours['profondeur'], 1):.1f}")
print(f"Contrôle : |E(circuit Qiskit) - E(simulateur interne)| = {RUN_INFO['ecart_simulateur_circuit']:.1e} Ha | "
      f"arrêt = {RUN_INFO['arret']} | {RUN_INFO['n_eval_E']} évaluations d'énergie | pool = {RUN_INFO['taille_pool']} opérateurs")
print("Ansatz :", RUN_INFO["ops"])

h = RUN_INFO["hist"]
plt.plot([e["n_ops"] for e in h], [max(1e3 * abs(e["E"] - exact_energy), 1e-6) for e in h], "o-", label="k-ADAPT")
plt.axhline(1e3 * precision_baseline, color="gray", ls=":", label="baseline UCCSD")
plt.axhline(1e3 * CHEMICAL_ACCURACY, color="green", ls="--", label="précision chimique")
plt.yscale("log"); plt.xlabel("Nombre d'opérateurs"); plt.ylabel("Erreur (mHa)")
plt.title(f"Convergence k-ADAPT — {MOLECULE} gelé"); plt.grid(alpha=0.3); plt.legend(); plt.show()

## 7. [Ajout équipe] Problème complet (sans gel de cœur)

In [ ]:
# [AJOUT ÉQUIPE] Même fonction run() sur le problème COMPLET (sans gel de cœur), comparé à exact_energy_full.
# Pas de baseline ici (UCCSD complet beaucoup trop lent avec la baseline de la consigne).
RUN_FULL = True
if RUN_FULL:
    e_full, t_full = run(problem_full, fermionic_op_full)
    err_full = abs(exact_energy_full - e_full)
    res_full = gate_counts(RUN_INFO["circuit"])
    print(f"[{MOLECULE} complet] {RUN_INFO['n_qubits']} qubits | E = {e_full:.6f} Ha | erreur = {1e3 * err_full:.4f} mHa "
          f"{'✅' if err_full <= CHEMICAL_ACCURACY else '❌'} | {t_full:.2f} s | {res_full['n_cnot']} CNOT, "
          f"{len(RUN_INFO['params'])} paramètres")
    print(f"Erreur due au gel de cœur (FCI gelé - FCI complet, énergies totales) : {1e3 * (exact_energy_frozen - exact_energy_full):.4f} mHa")